# 年度估值信息失败归因
冻结全部31个月和六个模型，仅对保存预测作精确线性与MSE恒等分解；不拟合、不重放、不开参数救援。

In [1]:
from pathlib import Path
import sys,os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
sys.path.insert(0,str(root))
import qlib
from qlib.data import D
from qlib.workflow import R
from src.analysis.annual_information_attribution import run
summary=run(root)
summary

2026/10/16 14:29:44 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[93388:MainThread](2026-10-16 14:29:46,439) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[93388:MainThread](2026-10-16 14:29:46,441) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[93388:MainThread](2026-10-16 14:29:46,442) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


[93388:MainThread](2026-10-16 14:29:46,698) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name etf_annual_information_attribution_20261016.


[93388:MainThread](2026-10-16 14:29:46,700) INFO - qlib.workflow - [exp.py:258] - Experiment 682201502985750691 starts running ...


[93388:MainThread](2026-10-16 14:29:46,785) INFO - qlib.workflow - [recorder.py:345] - Recorder b8944e07161645e9a2e2ffbd5780bf75 starts running under Experiment 682201502985750691 ...


[93388:MainThread](2026-10-16 14:29:48,040) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{'experiment': 'etf_annual_information_attribution_20261016',
 'decision': 'saved_forecast_failure_attribution_only_parent_remains_closed',
 'objective_achieved': False,
 'new_fits': 0,
 'new_inference': 0,
 'new_accounts': 0,
 'new_fetches': 0,
 'evaluation_months': 31,
 'head_observations': 62,
 'saved_models': 6,
 'maximum_prediction_identity_error': 0.0,
 'paired_normalized_mse_improvement': -0.18188343234227894,
 'mean_residual_delta_alignment': -0.0984336542717328,
 'covariance_alignment': -0.005630709201577455,
 'mean_shift_penalty': 0.04680617758007631,
 'delta_variation_penalty': 0.031012891288892376,
 'parent_ci_reused': {'paired_mean_loss_improvement': -0.18188343234227902,
  'lower_bound': -0.4648214117012018,
  'upper_bound': 0.06972794889800268,
  'valid_replicates': 2000,
  'invalid_replicates': 0,
  'calendar_rows': 31,
  'scored_rows': 31},
 'parent_gates_unchanged': {'pooled_improvement_positive': False,
  'bootstrap_lower_positive': False,
  'complete_year_70pct': Fa

# 年度估值信息：保存预测的失败归因

结论：两个预测头都变差，并非一好一坏互相抵消。候选预测整体向下移动，而控制预测的平均残差为正，即控制本来已经平均低估了实现值；新增预测位移进一步加重低估。这里是固定历史预测的会计分解，不是特征因果效应。

复用2024-01至2026-07全部31个月、两个标签、六个年度冻结Ridge模型及其训练变换。没有拟合、预测API调用、新标签、账户或来源抓取。存储的线性系数与变换逐项还原保存预测，最大误差为0；62条月度—标签恒等式及372条月度—特征组分配均闭合，并由独立评估者从Qlib Recorder读回。

每个标签使用原训练标准差，定义e=(实际−控制)/训练标准差，d=(候选−控制)/训练标准差。MSE改善恒等于2ed−d²。先分别分解两个标签，再等权合并：

|全部31个月的分量|归一MSE改善贡献|
|---|---:|
|平均残差与平均预测位移的方向配合|−0.098434|
|残差与预测位移的协变配合|−0.005631|
|平均位移平方代价|−0.046806|
|位移变化方差代价|−0.031013|
|合计|−0.181883|

SOXX收益头为−0.203459，SOXL相对SOXX收益头为−0.160308。上述量是训练方差归一的预测误差，不是收益率或资金损失比例。父卡区块区间仍为[−0.464821,0.069728]：没有可靠改善证据，也不能把跨零区间说成统计上已证明必然恶化。

|评价年度|月数|等权MSE改善|解释|
|---|---:|---:|---|
|2024|12|−0.048001|平均残差接近零，预测位移没有获得相应配合，位移代价占主要部分|
|2025|12|−0.201100|协变项+0.035917，但平均方向配合−0.134818，不能抵消位移代价|
|2026|7|−0.378454|平均方向与协变均不利；不完整年度，不能充当全年资格|

固定线性分组采用2e·d_g−d·d_g分配交叉代价，各组严格加总回父损失。全部月度的分配为：年度E/P−0.188120、已知亏损权重−0.053466、可选分位及缺失标记−0.046334、市场系数共同重估+0.039768、年度增长及缺失标记+0.066268、截距约0。这不是移除某组后重训的反事实，也不是独立特征价值；不能据此删除亏损组或分位来挽救已关闭版本。年度增长组在2026部分年也转为−0.049834，正的全期会计分配不等于稳定信息。

分位的实际启用过程是一个明确限制：2024年训练中没有有效分位值，数值坐标冻结为不活跃；虽然2024-09-30起原始分位开始可用，该年度分位组对保存预测仍为零。2025年训练38个月中只有2个有效分位观察，训练标准差0.012056，评价期标准化分位最大绝对值15.455；2026年有14个有效训练分位，最大绝对值降为1.950。2025年分位组的会计分配为−0.107230，但年度间模型、标签和其他系数同时变化，因此不能把损失增大全部归因于“启用分位”。年度E/P组的负分配更大，分位也不是唯一解释。

父实验保持not_promoted：总体改善、区块下界和完整年度门均未通过。全部月份、全部年度、全部季度和两个标签都已保留，没有挑选最差窗口、改阈值或做救援试验。历史均已用于研究，仍无独立前向或净账户超额证据。

下一步应是另行冻结不同且可证伪的信息机制、来源时点与配对检验；本卡不授权删特征、改正则、改变窗口或重训年度版本。关闭这个固定实验不等于结束稳定超额目标。

入口：plan.json、score_identity_summary.csv、feature_group_summary.csv、coefficient_summary.csv、feature_chronology.csv、summary.json、independent_evaluation.json。月度标签/分组明细、六个保存模型和父来源审计均在Recorder/NAS。

Recorder：`b8944e07161645e9a2e2ffbd5780bf75`；NAS SHA256：`d8ecdd96b8c673228d214bc18e86bda461a46e05d86af4f184c4b6acb8d29d1b`。
